# Intelligent Retail Decision Assistant

This notebook walks through all six phases of the capstone. The referenced starter CSV and policy JSON were not included with the brief, so this project generates a deterministic, clearly labeled synthetic dataset. All results and policy snippets are illustrative, not real business findings or legal advice.

Run from the `solution/` directory after installing `requirements.txt`. Optional OpenAI answer generation requires `OPENAI_API_KEY`; the default is fully offline.

## Phase 1 — Problem framing and dataset preparation

Business goal: predict purchase propensity from pre/post interaction context for demonstration, identify behavioral groupings, and support catalog/policy questions. This section generates the data, checks its schema, missingness, and target balance.

In [1]:
from pathlib import Path
import sys
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'capstone_graded_project' / 'final_Capstone Project _ Intelligent Retail Decision Assistant ' / 'solution']
solution_dir = next((path for path in candidates if (path / 'phase1_data_preparation.py').is_file()), None)
if solution_dir is None:
    raise RuntimeError('Run this notebook from the solution directory or repository root.')
sys.path.insert(0, str(solution_dir))
from phase1_data_preparation import prepare_dataset, load_interactions
profile = prepare_dataset()
display(profile)
interactions = load_interactions()
display(interactions.head())
display(interactions.isna().sum().loc[lambda s: s.gt(0)])

{'rows': 1200,
 'columns': 41,
 'missing_values': {'rating_given': 516,
  'customer_review_text': 516,
  'image_path': 1200},
 'purchase_flag_balance': {'0': 516, '1': 684},
 'source_note': 'Synthetic deterministic data generated for this submission; the referenced source CSV was not included.'}

,customer_id,age,income_band,membership_level,city_tier,total_orders,avg_order_value,days_since_last_purchase,preferred_channel,product_id,...,purchase_flag,quantity,returned_flag,rating_given,customer_review_text,customer_query,policy_doc_id,policy_text_snippet,occasion_tags,image_path
0,C0082,25,mid,silver,tier_1,7,5434.01,94,app,P008,...,0,0,0,NaN,NaN,Looking for grey sports shoes for gym,RET_01,Unused products may be returned within 7 calen...,gym training sports running,NaN
1,C0087,32,value,gold,tier_1,4,2433.01,50,app,P005,...,0,0,0,NaN,NaN,Looking for black formal shoes for formal,RET_01,Unused products may be returned within 7 calen...,formal office business wedding,NaN
2,C0134,56,premium,standard,tier_1,8,1636.57,90,web,P014,...,1,2,0,3.0,Looks as described and feels durable.,Looking for white shirts for formal,RET_01,Unused products may be returned within 7 calen...,formal office business,NaN
3,C0151,31,premium,gold,tier_3,7,2869.33,168,app,P016,...,1,1,0,3.0,Looks as described and feels durable.,Looking for black jackets for running,RET_01,Unused products may be returned within 7 calen...,running sports outdoor,NaN
4,C0076,33,mid,gold,tier_3,1,827.85,121,store,P022,...,0,0,0,NaN,NaN,Looking for black clogs for casual,RET_01,Unused products may be returned within 7 calen...,casual walking daily,NaN


rating_given             516
customer_review_text     516
image_path              1200
dtype: int64

## Phase 2 — Classical ML pipeline

A class-weighted logistic-regression baseline uses median/mode imputation, standardization for numeric fields, and one-hot encoding for categorical fields. The holdout is stratified. Post-purchase data are excluded from purchase prediction. K-Means segmentation chooses the best silhouette score for k=2..5.

In [2]:
from phase2_classical_ml import run_classical_ml
ml_results = run_classical_ml()
display(ml_results['supervised'])
display(ml_results['important_features'])
display(ml_results['segmentation'])

{'test_rows': 240,
 'positive_rate': 0.57,
 'accuracy': 0.5917,
 'precision': 0.6789,
 'recall': 0.5401,
 'f1': 0.6016,
 'roc_auc': 0.6231,
 'baseline': 'class-weighted logistic regression; 80/20 stratified holdout, random_state=42'}

[{'feature': 'numeric__added_to_cart',
  'coefficient': 0.7279199861715772,
  'absolute_coefficient': 0.7279199861715772},
 {'feature': 'categorical__sub_category_Trousers',
  'coefficient': -0.312190064129892,
  'absolute_coefficient': 0.312190064129892},
 {'feature': 'numeric__discount_percent',
  'coefficient': 0.27035234002471165,
  'absolute_coefficient': 0.27035234002471165},
 {'feature': 'categorical__sub_category_Shirts',
  'coefficient': 0.23525228043089266,
  'absolute_coefficient': 0.23525228043089266},
 {'feature': 'categorical__color_Grey',
  'coefficient': -0.2270490232096665,
  'absolute_coefficient': 0.2270490232096665},
 {'feature': 'categorical__sub_category_Formal Shoes',
  'coefficient': -0.20784162761861133,
  'absolute_coefficient': 0.20784162761861133},
 {'feature': 'categorical__material_Polyester',
  'coefficient': -0.19882017333703048,
  'absolute_coefficient': 0.19882017333703048},
 {'feature': 'categorical__preferred_channel_web',
  'coefficient': 0.18698039

{'clusters': 4,
 'silhouette_score': 0.132,
 'unit_of_analysis': 'customer profiles aggregated from their interaction rows',
 'features': ['days_since_last_purchase',
  'avg_order_value',
  'total_orders',
  'final_price',
  'discount_percent',
  'time_on_page',
  'wishlist_flag'],
 'distinguishing_feature': 'avg_order_value',
 'segments': {'0': 'lower historical order value (cluster 0)',
  '1': 'lower historical order value (cluster 1)',
  '2': 'lower historical order value (cluster 2)',
  '3': 'higher historical order value (cluster 3)'},
 'segment_details': {'0': {'customers': 40.0,
   'mean_interactions': 6.08,
   'mean_final_price': 2604.19,
   'mean_discount_percent': 11.38,
   'mean_avg_order_value': 1762.11,
   'mean_days_since_last_purchase': 143.98,
   'mean_total_orders': 7.05,
   'mean_time_on_page': 73.77,
   'wishlist_rate': 0.32},
  '1': {'customers': 59.0,
   'mean_interactions': 5.9,
   'mean_final_price': 3232.84,
   'mean_discount_percent': 11.58,
   'mean_avg_order_

## Phase 3 — Enrich data for Generative AI

Product text combines title, description, brand, category, tags, and price. Policy evidence is authored for this synthetic exercise and assigned a stable document ID.

In [3]:
from phase3_retrieval_corpus import prepare_retrieval_corpus, build_product_documents, build_policy_documents
corpus_counts = prepare_retrieval_corpus()
display(corpus_counts)
display(build_product_documents()[:3])
display(build_policy_documents()[:2])

{'products': 24, 'policies': 6}

[{'product_id': 'P001',
  'product_title': 'Everyday Mesh Runner',
  'brand': 'Stride',
  'category': 'Shoes',
  'style_tags': 'casual walking daily sports',
  'base_price': 2799.0,
  'text': 'Everyday Mesh Runner. Everyday Mesh Runner by Stride. Mesh sneakers in black. Ideal for casual walking daily sports. Brand Stride. Category Shoes. Style and occasions: casual walking daily sports. Price ₹2799.'},
 {'product_id': 'P002',
  'product_title': 'Cloudfoam Walking Sneaker',
  'brand': 'Northstar',
  'category': 'Shoes',
  'style_tags': 'casual walking daily travel',
  'base_price': 3499.0,
  'text': 'Cloudfoam Walking Sneaker. Cloudfoam Walking Sneaker by Northstar. Knit sneakers in white. Ideal for casual walking daily travel. Brand Northstar. Category Shoes. Style and occasions: casual walking daily travel. Price ₹3499.'},
 {'product_id': 'P003',
  'product_title': 'Urban Canvas Low-Top',
  'brand': 'Streetwise',
  'category': 'Shoes',
  'style_tags': 'casual everyday travel',
  'base

[{'policy_doc_id': 'RET_01',
  'topic': 'returns',
  'title': 'Returns',
  'policy_text': 'Unused products may be returned within 7 calendar days of delivery. Items must retain original tags and packaging. Products marked final sale are not returnable.'},
 {'policy_doc_id': 'RET_02',
  'topic': 'discounted returns',
  'title': 'Discounted item returns',
  'policy_text': 'Discounted products can be returned within 7 calendar days of delivery if unused and in original packaging with tags attached. After 7 days they are not eligible for return.'}]

## Phase 4 — Retrieval and grounded answers

Local TF-IDF vectors feed an exact FAISS inner-product index when installed, or a scikit-learn cosine-search fallback. Policy answers use retrieved text only, include policy IDs, and abstain if evidence is weak. Product search enforces an explicitly stated rupee budget.

In [4]:
from phase4_rag import RetailKnowledgeBase
knowledge_base = RetailKnowledgeBase()
display(knowledge_base.search_products('casual sneakers under ₹4000 daily wear'))
display(knowledge_base.answer_products('Recommend casual sneakers under ₹4000 for daily wear', use_llm=False))
display(knowledge_base.answer_policy('Can I return a discounted product after 7 days?', use_llm=False))

[{'product_id': 'P024',
  'product_title': 'Cushioned Daily Trainer',
  'brand': 'Stride',
  'category': 'Shoes',
  'style_tags': 'casual walking running daily',
  'base_price': 3299.0,
  'text': 'Cushioned Daily Trainer. Cushioned Daily Trainer by Stride. Mesh sneakers in grey. Ideal for casual walking running daily. Brand Stride. Category Shoes. Style and occasions: casual walking running daily. Price ₹3299.',
  'relevance_score': 0.3043},
 {'product_id': 'P002',
  'product_title': 'Cloudfoam Walking Sneaker',
  'brand': 'Northstar',
  'category': 'Shoes',
  'style_tags': 'casual walking daily travel',
  'base_price': 3499.0,
  'text': 'Cloudfoam Walking Sneaker. Cloudfoam Walking Sneaker by Northstar. Knit sneakers in white. Ideal for casual walking daily travel. Brand Northstar. Category Shoes. Style and occasions: casual walking daily travel. Price ₹3499.',
  'relevance_score': 0.2207},
 {'product_id': 'P001',
  'product_title': 'Everyday Mesh Runner',
  'brand': 'Stride',
  'cate

{'answer': 'Retrieved catalog evidence: Cushioned Daily Trainer (P024, ₹3299) — Cushioned Daily Trainer. Cushioned Daily Trainer by Stride. Mesh sneakers in grey. Ideal for casual walking running daily. Brand Stride. Category Shoes. Style and occasions: casual walking running daily. Price ₹3299. Cloudfoam Walking Sneaker (P002, ₹3499) — Cloudfoam Walking Sneaker. Cloudfoam Walking Sneaker by Northstar. Knit sneakers in white. Ideal for casual walking daily travel. Brand Northstar. Category Shoes. Style and occasions: casual walking daily travel. Price ₹3499. Everyday Mesh Runner (P001, ₹2799) — Everyday Mesh Runner. Everyday Mesh Runner by Stride. Mesh sneakers in black. Ideal for casual walking daily sports. Brand Stride. Category Shoes. Style and occasions: casual walking daily sports. Price ₹2799.',
 'products': [{'product_id': 'P024',
   'product_title': 'Cushioned Daily Trainer',
   'brand': 'Stride',
   'category': 'Shoes',
   'style_tags': 'casual walking running daily',
   'bas

{'answer': 'Discounted products can be returned within 7 calendar days of delivery if unused and in original packaging with tags attached. After 7 days they are not eligible for return. [Source: RET_02]',
 'evidence': [{'policy_doc_id': 'RET_02',
   'text': 'Discounted products can be returned within 7 calendar days of delivery if unused and in original packaging with tags attached. After 7 days they are not eligible for return.',
   'relevance_score': 0.4236}],
 'sources': ['RET_02']}

## Phase 5 — Agent/controller

Explicit keyword rules route to one tool: policy search, product search, or descriptive analytics. The controller returns its decision and rationale as a trace.

In [5]:
from phase5_controller import RetailAssistant
assistant = RetailAssistant(knowledge_base)
for query in [
    'Recommend casual sneakers under ₹4000 for daily wear',
    'Can I return a discounted product after 7 days?',
    'Suggest products for a premium customer who prefers black formal shoes',
    'Which customer segment seems most price-sensitive?',
    'Summarise the top three casual daily-wear products and explain why they fit',
]:
    display(assistant.answer(query))

{'route': 'product_search',
 'trace': [{'step': 1,
   'decision': 'product_search',
   'reason': 'No policy or analytics keyword matched; search the product catalog.'}],
 'answer': 'Retrieved catalog evidence: Cushioned Daily Trainer (P024, ₹3299) — Cushioned Daily Trainer. Cushioned Daily Trainer by Stride. Mesh sneakers in grey. Ideal for casual walking running daily. Brand Stride. Category Shoes. Style and occasions: casual walking running daily. Price ₹3299. Cloudfoam Walking Sneaker (P002, ₹3499) — Cloudfoam Walking Sneaker. Cloudfoam Walking Sneaker by Northstar. Knit sneakers in white. Ideal for casual walking daily travel. Brand Northstar. Category Shoes. Style and occasions: casual walking daily travel. Price ₹3499. Everyday Mesh Runner (P001, ₹2799) — Everyday Mesh Runner. Everyday Mesh Runner by Stride. Mesh sneakers in black. Ideal for casual walking daily sports. Brand Stride. Category Shoes. Style and occasions: casual walking daily sports. Price ₹2799.',
 'products': [{'

{'route': 'policy_search',
 'trace': [{'step': 1,
   'decision': 'policy_search',
   'reason': 'Policy-related keyword matched; retrieve policy evidence only.'}],
 'answer': 'Discounted products can be returned within 7 calendar days of delivery if unused and in original packaging with tags attached. After 7 days they are not eligible for return. [Source: RET_02]',
 'evidence': [{'policy_doc_id': 'RET_02',
   'text': 'Discounted products can be returned within 7 calendar days of delivery if unused and in original packaging with tags attached. After 7 days they are not eligible for return.',
   'relevance_score': 0.4236}],
 'sources': ['RET_02']}

{'route': 'product_search',
 'trace': [{'step': 1,
   'decision': 'product_search',
   'reason': 'No policy or analytics keyword matched; search the product catalog.'}],
 'answer': 'Retrieved catalog evidence: Premium Suede Chelsea Boot (P023, ₹6999) — Premium Suede Chelsea Boot. Premium Suede Chelsea Boot by Marlow. Suede boots in black. Ideal for formal office winter. Brand Marlow. Category Shoes. Style and occasions: formal office winter. Price ₹6999. Classic Leather Formal Oxford (P005, ₹4999) — Classic Leather Formal Oxford. Classic Leather Formal Oxford by Regent. Leather formal shoes in black. Ideal for formal office business wedding. Brand Regent. Category Shoes. Style and occasions: formal office business wedding. Price ₹4999. Office Comfort Derby (P006, ₹3799) — Office Comfort Derby. Office Comfort Derby by Regent. Leather formal shoes in black. Ideal for formal office business. Brand Regent. Category Shoes. Style and occasions: formal office business. Price ₹3799.',
 'produc

{'route': 'retail_analytics',
 'trace': [{'step': 1,
   'decision': 'retail_analytics',
   'reason': 'Analytics keyword matched; report the reproducible segmentation summary.'}],
 'answer': "Segment 3 had the highest observed average discount (15.0%). The other segment's mean was 11.4%. This is discount exposure in synthetic interactions, not evidence of causal price sensitivity. Segments are described by avg_order_value; silhouette score: 0.132.",
 'sources': ['outputs/segmentation_summary.json',
  'outputs/segment_summary.csv']}

{'route': 'product_search',
 'trace': [{'step': 1,
   'decision': 'product_search',
   'reason': 'No policy or analytics keyword matched; search the product catalog.'}],
 'answer': 'Retrieved catalog evidence: Urban Canvas Low-Top (P003, ₹1899) — Urban Canvas Low-Top. Urban Canvas Low-Top by Streetwise. Canvas sneakers in black. Ideal for casual everyday travel. Brand Streetwise. Category Shoes. Style and occasions: casual everyday travel. Price ₹1899. Cushioned Daily Trainer (P024, ₹3299) — Cushioned Daily Trainer. Cushioned Daily Trainer by Stride. Mesh sneakers in grey. Ideal for casual walking running daily. Brand Stride. Category Shoes. Style and occasions: casual walking running daily. Price ₹3299. Everyday Sandal (P012, ₹1299) — Everyday Sandal. Everyday Sandal by Coast. Synthetic sandals in tan. Ideal for casual summer daily travel. Brand Coast. Category Shoes. Style and occasions: casual summer daily travel. Price ₹1299.',
 'products': [{'product_id': 'P003',
   'product_title

## Final phase — Evaluation and packaging

Run the end-to-end regression checks and inspect persisted metrics, segment summaries, and example answers in `outputs/`.

In [6]:
from phase6_evaluation import run_evaluation
final_report = run_evaluation()
display(final_report['regression_checks'])
display(final_report['phase2'])
display(final_report['phase5_queries'])

{'five_queries': True,
 'product_budget_respected': True,
 'policy_source_cited': True,
 'controller_routes_policy': True,
 'controller_routes_analytics': True}

{'supervised_metrics': {'test_rows': 240,
  'positive_rate': 0.57,
  'accuracy': 0.5917,
  'precision': 0.6789,
  'recall': 0.5401,
  'f1': 0.6016,
  'roc_auc': 0.6231,
  'baseline': 'class-weighted logistic regression; 80/20 stratified holdout, random_state=42'},
 'segmentation': {'clusters': 4,
  'silhouette_score': 0.132,
  'unit_of_analysis': 'customer profiles aggregated from their interaction rows',
  'features': ['days_since_last_purchase',
   'avg_order_value',
   'total_orders',
   'final_price',
   'discount_percent',
   'time_on_page',
   'wishlist_flag'],
  'distinguishing_feature': 'avg_order_value',
  'segments': {'0': 'lower historical order value (cluster 0)',
   '1': 'lower historical order value (cluster 1)',
   '2': 'lower historical order value (cluster 2)',
   '3': 'higher historical order value (cluster 3)'},
  'segment_details': {'0': {'customers': 40.0,
    'mean_interactions': 6.08,
    'mean_final_price': 2604.19,
    'mean_discount_percent': 11.38,
    'mean_

[{'route': 'product_search',
  'trace': [{'step': 1,
    'decision': 'product_search',
    'reason': 'No policy or analytics keyword matched; search the product catalog.'}],
  'answer': 'Retrieved catalog evidence: Cushioned Daily Trainer (P024, ₹3299) — Cushioned Daily Trainer. Cushioned Daily Trainer by Stride. Mesh sneakers in grey. Ideal for casual walking running daily. Brand Stride. Category Shoes. Style and occasions: casual walking running daily. Price ₹3299. Cloudfoam Walking Sneaker (P002, ₹3499) — Cloudfoam Walking Sneaker. Cloudfoam Walking Sneaker by Northstar. Knit sneakers in white. Ideal for casual walking daily travel. Brand Northstar. Category Shoes. Style and occasions: casual walking daily travel. Price ₹3499. Everyday Mesh Runner (P001, ₹2799) — Everyday Mesh Runner. Everyday Mesh Runner by Stride. Mesh sneakers in black. Ideal for casual walking daily sports. Brand Stride. Category Shoes. Style and occasions: casual walking daily sports. Price ₹2799.',
  'products